# Real or noise, cause or coincidence

**Kalpa Retail, Week 1 Day 4.** With Finance reconciled, Meera sets the growth review for Monday
and sends three questions.

> "One: Retail-Plus is down, smaller than first reported. Real, or the wobble we see every
> quarter? Two: Student is up 40 percent; should I move budget there? Three: marketing ran a
> monsoon-sale discount for Retail-Plus in August, says it lifted revenue 6 percent, and wants to
> repeat it for Diwali. Did the discount work, or did those customers buy anyway?"

Her constraint: **"One page, two minutes. If the honest answer is 'we do not know yet', say so and
tell me what would tell us."**

Three questions that look alike and need three different habits.

In [1]:
import pathlib
import random
import sys

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

ORDERS = kit.load_csv("C2_W01_D04_orders_STUDENT.csv")
EXPOSURE = kit.load_csv("C2_W01_D04_exposure_STUDENT.csv")
print(f"{len(ORDERS)} cleaned orders and {len(EXPOSURE)} customers in the campaign table")

186 cleaned orders and 160 customers in the campaign table


## MAP: three questions, three habits

In [2]:
kit.tree(
    {"label": "Meera's three questions",
     "branches": [
         ("real or the wobble", {"label": "a chance reference"}),
         ("fund Student", {"label": "sample size"}),
         ("did the discount work", {"label": "a fair comparison"}),
     ]},
    taken=["real or the wobble"],
    title="They look alike. Answering one with another's method is today's failure mode.")

## Question one: is the Retail-Plus fall real?

The fall is 35 percent against Retail-Core's 2.7, a gap of 32.3 points. **Suppose the two labels
meant nothing** and the same members had been split between them at random. Would a gap that large
turn up anyway?

In [3]:
def fall(rows):
    q1 = [r for r in rows if r["quarter"] == "Q1"]
    q2 = [r for r in rows if r["quarter"] == "Q2"]
    a = len(q1) / len({r["customer_id"] for r in q1})
    b = len(q2) / len({r["customer_id"] for r in q2})
    return 100 * (b / a - 1)


plus = [r for r in ORDERS if r["segment"] == "Retail-Plus"]
core = [r for r in ORDERS if r["segment"] == "Retail-Core"]
observed = fall(core) - fall(plus)
print(f"Retail-Plus fell {fall(plus):.1f} percent")
print(f"Retail-Core fell {fall(core):.1f} percent")
print(f"the gap:         {observed:.1f} points")

Retail-Plus fell -35.0 percent
Retail-Core fell -2.7 percent
the gap:         32.3 points


### The shuffle, ten times first

Shuffle which members are labelled Plus and which Core, keeping the group sizes, then recompute the
gap. Ten of them is enough to see what the move is.

In [4]:
by_customer = {}
for r in plus + core:
    by_customer.setdefault(r["customer_id"], []).append(r)
members = sorted(by_customer)
n_plus = len({r["customer_id"] for r in plus})


def shuffled_gap(rng):
    m = members[:]
    rng.shuffle(m)
    a = [r for c in m[:n_plus] for r in by_customer[c]]
    b = [r for c in m[n_plus:] for r in by_customer[c]]
    return fall(b) - fall(a)


rng = random.Random(20260101)
ten = [shuffled_gap(rng) for _ in range(10)]
kit.table(["shuffle", "gap chance produced"],
          [(i + 1, f"{g:+.1f} points") for i, g in enumerate(ten)],
          caption=f"Ten chance-only worlds, against a real gap of {observed:.1f} points")

shuffle,gap chance produced
1,+4.1 points
2,-0.2 points
3,+0.9 points
4,+3.1 points
5,+2.0 points
6,-1.2 points
7,+3.1 points
8,-8.2 points
9,+14.8 points
10,-4.6 points


### Now five thousand of them

In [5]:
rng = random.Random(20260101)
N = 5000
extreme = sum(1 for _ in range(N) if abs(shuffled_gap(rng)) >= abs(observed))
print(f"{extreme} of {N} shuffles reached {abs(observed):.1f} points or more")
if extreme:
    print(f"p = {extreme / N:.4f}")
else:
    print(f"p < {1 / N:.4f}")

0 of 5000 shuffles reached 32.3 points or more
p < 0.0002


### Never write `p = 0`

Five thousand shuffles can only resolve down to one in five thousand. Writing `p = 0` claims a
certainty the method cannot produce, so the honest report is **`p < 0.0002`**: the resolution of
your own simulation.

In [6]:
kit.check("the observed gap is about 32 points", 30 < abs(observed) < 35,
          f"{observed:.1f} points")
kit.check("chance alone does not reach it", extreme == 0, f"{extreme} of {N}")

### What the p-value is, and what it is not

It is **the share of chance-only worlds that produce a result at least this extreme**. It is not
the probability the finding is wrong, it is not the probability chance caused it, and it says
nothing about how big the effect is.

In [7]:
kit.vflow(["could chance have done this?  the p-value",
           "is it big enough to care about?  the size of the effect",
           "is it worth doing something about?  the cost against the gain"],
          lit=0, title="Three separate calls, and only the first came from the shuffle")

**Retail-Plus, all three calls.** Chance: no, `p < 0.0002`. Big: yes, orders per member fell about
a third. Worth acting on: these are 22 paid-tier members, so yes. Only the first came from the
data; the other two came from knowing the business.

## Question two: should Meera fund Student?

In [8]:
student = [r for r in ORDERS if r["segment"] == "Student"]
s1 = [r for r in student if r["quarter"] == "Q1"]
s2 = [r for r in student if r["quarter"] == "Q2"]
print(f"Student orders: {len(student)} in total, {len(s1)} in Q1 and {len(s2)} in Q2")
print(f"orders per member: {fall(student):+.1f} percent")

Student orders: 12 in total, 5 in Q1 and 7 in Q2
orders per member: +40.0 percent


Forty percent, on twelve orders. **Suppose each of those twelve orders landed in either quarter by
chance.** How often does that alone give seven or more in the second?

In [9]:
rng = random.Random(20260101)
ratio = len(s2) / len(s1)
hits = 0
for _ in range(N):
    a = sum(1 for _ in range(len(student)) if rng.random() < 0.5)
    b = len(student) - a
    if a and b / a >= ratio:
        hits += 1
print(f"{hits} of {N} chance-only worlds were at least this extreme")
print(f"p = {hits / N:.3f}")

1914 of 5000 chance-only worlds were at least this extreme
p = 0.383


In [10]:
kit.check("Student rests on twelve orders", len(student) == 12, f"{len(student)}")
kit.check("chance produces the Student rise often", hits / N > 0.2, f"p = {hits / N:.3f}")
kit.check("the two verdicts are opposite", (extreme / N) < 0.01 < (hits / N),
          f"Retail-Plus p < 0.0002 against Student p = {hits / N:.3f}")

**The rule of thumb.** Distrust any rate computed on fewer than about thirty observations. It is a
rule of thumb rather than a law, and saying which it is out loud is part of using it honestly.
Student has twelve; Retail-Plus has sixty-six.

## Question three: did the monsoon sale work?

Marketing's claim is arithmetically correct, and it is the most interesting number of the week.

In [11]:
def spend(exposed, segment=None):
    rows = [r for r in EXPOSURE if r["exposed"] == exposed
            and (segment is None or r["segment"] == segment)]
    return sum(int(r["august_revenue"]) for r in rows) / len(rows), len(rows)


blended_yes, n_yes = spend("yes")
blended_no, n_no = spend("no")
print(f"exposed:     Rs {blended_yes:,.0f} per customer across {n_yes}")
print(f"not exposed: Rs {blended_no:,.0f} per customer across {n_no}")
print(f"marketing's claim: {100 * (blended_yes / blended_no - 1):+.1f} percent")

exposed:     Rs 3,395 per customer across 60
not exposed: Rs 3,200 per customer across 100
marketing's claim: +6.1 percent


### Now split it by segment

In [12]:
rows = []
for seg in ("Retail-Plus", "Retail-Core"):
    y, ny = spend("yes", seg)
    n, nn = spend("no", seg)
    rows.append((seg, f"Rs {y:,.0f} ({ny})", f"Rs {n:,.0f} ({nn})",
                 f"{100 * (y / n - 1):+.1f}%"))
rows.append(("Everyone", f"Rs {blended_yes:,.0f} ({n_yes})", f"Rs {blended_no:,.0f} ({n_no})",
             f"{100 * (blended_yes / blended_no - 1):+.1f}%"))
kit.table(["group", "exposed", "not exposed", "change"], rows,
          caption="Both parts fell. The whole rose. Nobody made an error.")

group,exposed,not exposed,change
Retail-Plus,"Rs 4,850 (30)","Rs 5,000 (40)",-3.0%
Retail-Core,"Rs 1,940 (30)","Rs 2,000 (60)",-3.0%
Everyone,"Rs 3,395 (60)","Rs 3,200 (100)",+6.1%


### Where the six percent came from

In [13]:
mix_yes = spend("yes", "Retail-Plus")[1] / n_yes
mix_no = spend("no", "Retail-Plus")[1] / n_no
print(f"Retail-Plus share of the exposed group:     {100 * mix_yes:.0f} percent")
print(f"Retail-Plus share of the control group:     {100 * mix_no:.0f} percent")
print(f"Retail-Plus spends about {spend('no', 'Retail-Plus')[0] / spend('no', 'Retail-Core')[0]:.1f} "
      f"times what Retail-Core spends")

kit.flow(["marketing targeted Retail-Plus",
          "so the exposed group is richer",
          "the blend is pulled upward",
          "+6 percent from the mix, not the discount"],
         lit=3, title="The campaign changed who is in the average")

Retail-Plus share of the exposed group:     50 percent
Retail-Plus share of the control group:     40 percent
Retail-Plus spends about 2.5 times what Retail-Core spends


In [14]:
kit.check("every segment fell", all(spend("yes", s)[0] < spend("no", s)[0]
                                    for s in ("Retail-Plus", "Retail-Core")),
          "both segments spent less when exposed")
kit.check("the blend rose anyway", blended_yes > blended_no,
          f"Rs {blended_yes:,.0f} against Rs {blended_no:,.0f}")
kit.check("the exposed group skews toward the richer segment", mix_yes > mix_no,
          f"{100 * mix_yes:.0f} percent against {100 * mix_no:.0f}")

**What you cannot say either.** The campaign did not "reduce spending by 3 percent". Nobody
randomised it, so the exposed and unexposed groups may differ in ways the segment split does not
capture. Over-correcting is as wrong as the claim it replaces.

**What would settle it:** assign the next discount at random within a segment, so the two groups
differ only in the discount. An experiment nobody ran cannot be recovered from the data afterwards.

In [15]:
kit.check_summary()

## The page Meera gets

> **Retail-Plus.** Orders per member fell about a third, against Retail-Core's 2.7 percent. Chance
> alone produced a gap this large in none of 5,000 shuffles, so the fall is real. These are 22
> paid-tier members and I would act on it.
>
> **Student.** Up 40 percent on twelve orders. Chance produces a rise that large about two times in
> five, so I would not move budget yet. A full quarter at around fifty orders would tell us.
>
> **The monsoon sale.** The six percent is a mix effect: half the exposed group is Retail-Plus
> against forty percent of the control, and Retail-Plus spends two and a half times more. Within
> both segments, exposed customers spent three percent **less**. I cannot say the campaign failed
> either, because nobody randomised it. Repeating it as designed is a bet with no evidence behind
> it; randomising the next one inside a segment would settle it.

Three answers, three different shapes, and only one of them is a yes. A page where all three are
yes is a page that was written to please.